In [1]:
import warnings
from pathlib import Path

import numpy as np
from keras import layers
import keras
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")
np.random.seed(42)
keras.utils.set_random_seed(42)

MODEL_DIR = Path("../models")
MODEL_DIR.mkdir(exist_ok=True)

FRAME_SIZE = 128
INPUT_SHAPE = (FRAME_SIZE, FRAME_SIZE, 3)
print(f"Frame input shape: {INPUT_SHAPE}")


def generate_genuine_frames(n=1):
    frames = []
    for _ in range(n):
        skin_r = np.random.uniform(0.6, 0.85)
        skin_g = np.random.uniform(0.45, 0.65)
        skin_b = np.random.uniform(0.35, 0.55)

        frame = np.zeros((FRAME_SIZE, FRAME_SIZE, 3), dtype=np.float32)
        frame[:, :, 0] = skin_r
        frame[:, :, 1] = skin_g
        frame[:, :, 2] = skin_b

        grad = np.linspace(0.8, 1.1, FRAME_SIZE, dtype=np.float32)[:, np.newaxis]
        frame *= grad[..., np.newaxis]
        frame += np.random.normal(0, 0.02, frame.shape).astype(np.float32)

        cx, cy = FRAME_SIZE // 2, FRAME_SIZE // 2
        for ex in [cx - 20, cx + 20]:
            ey = cy - 15
            frame[max(0, ey - 5):ey + 5, max(0, ex - 8):ex + 8, :] *= 0.4
        frame[cy + 15:cy + 22, cx - 15:cx + 15, :] *= 0.5

        frames.append(np.clip(frame, 0, 1))
    return np.array(frames, dtype=np.float32)


def generate_deepfake_frames(n=1):
    frames = []
    for _ in range(n):
        frame = generate_genuine_frames(1)[0]

        boundary_r = np.random.randint(35, 55)
        cy, cx = FRAME_SIZE // 2, FRAME_SIZE // 2
        y, x = np.ogrid[:FRAME_SIZE, :FRAME_SIZE]
        mask = ((y - cy) ** 2 + (x - cx) ** 2) > boundary_r ** 2
        frame[mask] *= np.random.uniform(0.85, 0.95)
        frame[mask] += np.random.uniform(0.03, 0.08)

        seam_y = cy + np.random.randint(-10, 10)
        seam_width = np.random.randint(2, 5)
        frame[seam_y:seam_y + seam_width, :, :] += np.random.uniform(0.05, 0.15)

        if np.random.random() > 0.5:
            frame[20:50, 20:50, :] = np.clip(frame[20:50, 20:50, :] * 1.3, 0, 1)

        frames.append(np.clip(frame, 0, 1))
    return np.array(frames, dtype=np.float32)


N_SAMPLES = 240
X_genuine = generate_genuine_frames(N_SAMPLES)
X_deepfake = generate_deepfake_frames(N_SAMPLES)

X = np.concatenate([X_genuine, X_deepfake], axis=0).astype(np.float32)
y = np.concatenate([np.zeros(N_SAMPLES), np.ones(N_SAMPLES)])

idx = np.random.permutation(len(X))
X, y = X[idx], y[idx]
print(f"Dataset: {X.shape}, Labels: {y.shape}")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {X_train.shape[0]}, Test: {X_test.shape[0]}")


def build_video_cnn(input_shape):
    model = keras.Sequential([
        layers.Input(shape=input_shape),
        layers.Conv2D(32, (3, 3), padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(64, (3, 3), padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(128, (3, 3), padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.GlobalAveragePooling2D(),
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.4),
        layers.Dense(1, activation="sigmoid"),
    ])

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss="binary_crossentropy",
        metrics=["accuracy", keras.metrics.AUC(name="auc")],
    )
    return model


model = build_video_cnn(INPUT_SHAPE)
model.summary()

callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_auc", mode="max", patience=2, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=1, min_lr=1e-6),
]

history = model.fit(
    X_train,
    y_train,
    validation_split=0.15,
    epochs=6,
    batch_size=32,
    callbacks=callbacks,
    verbose=1,
)

y_prob = model.predict(X_test, verbose=0).flatten()
y_pred = (y_prob >= 0.5).astype(int)

print("=== Video Deepfake Detection Results ===")
print(classification_report(y_test, y_pred, target_names=["Genuine", "Deepfake"]))
print(f"AUC-ROC: {roc_auc_score(y_test, y_prob):.4f}")

model_path = MODEL_DIR / "video_deepfake_detector.h5"
model.save(model_path)
print(f"Model saved: {model_path} ({model_path.stat().st_size / 1024 / 1024:.1f} MB)")
print("Video deepfake model training complete.")

Frame input shape: (128, 128, 3)
Dataset: (480, 128, 128, 3), Labels: (480,)
Train: 384, Test: 96


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 128, 128, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 128, 128, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 64, 64, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 64, 64, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 64, 64, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 32, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 32, 32, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 32, 32, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 16, 16, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 110,785 (432.75 KB)

 Trainable params: 110,337 (431.00 KB)

 Non-trainable params: 448 (1.75 KB)

Epoch 1/6
11/11 ━━━━━━━━━━━━━━━━━━━━ 8s 400ms/step - accuracy: 0.4908 - auc: 0.4989 - loss: 0.8309 - val_accuracy: 0.5000 - val_auc: 0.5000 - val_loss: 0.6931 - learning_rate: 0.0010
Epoch 2/6
11/11 ━━━━━━━━━━━━━━━━━━━━ 4s 398ms/step - accuracy: 0.5307 - auc: 0.5413 - loss: 0.7472 - val_accuracy: 0.5000 - val_auc: 0.5309 - val_loss: 0.6941 - learning_rate: 0.0010
Epoch 3/6
11/11 ━━━━━━━━━━━━━━━━━━━━ 4s 399ms/step - accuracy: 0.5368 - auc: 0.5618 - loss: 0.7244 - val_accuracy: 0.5000 - val_auc: 0.5446 - val_loss: 0.6967 - learning_rate: 5.0000e-04
Epoch 4/6
11/11 ━━━━━━━━━━━━━━━━━━━━ 5s 411ms/step - accuracy: 0.5736 - auc: 0.6257 - loss: 0.6799 - val_accuracy: 0.5000 - val_auc: 0.5470 - val_loss: 0.7025 - learning_rate: 2.5000e-04
Epoch 5/6
11/11 ━━━━━━━━━━━━━━━━━━━━ 5s 406ms/step - accuracy: 0.6135 - auc: 0.6525 - loss: 0.6539 - val_accuracy: 0.5000 - val_auc: 0.5416 - val_loss: 0.7112 - learning_rate: 1.2500e-04
Epoch 6/6
11/11 ━━━━━━━━━━━━━━━━━━━━ 4s 377ms/step - accuracy: 0.6012 - a

=== Video Deepfake Detection Results ===
              precision    recall  f1-score   support

     Genuine       0.00      0.00      0.00        48
    Deepfake       0.50      1.00      0.67        48

    accuracy                           0.50        96
   macro avg       0.25      0.50      0.33        96
weighted avg       0.25      0.50      0.33        96

AUC-ROC: 0.5686
Model saved: ..\models\video_deepfake_detector.h5 (1.3 MB)
Video deepfake model training complete.
